### Step 1: Install and Import Necessary Libraries

In [ ]:
%pip install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

### Step 2: Load the Sentence Embedding Model

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded successfully!")

### Step 3: Define Documents and Sample Query

In [ ]:
documents = [
    "Students can apply for scholarships and financial aid.",
    "Python is a popular programming language used in AI.",
    "The university library is open from 9AM to 8PM.",
    "Students can complete their internship during their studies.",
    "Machine learning allows computers to learn from data.",
    "The university offers courses in Python programming, machine learning, and artificial intelligence.",
    "Students must complete all required examinations and credits to graduate.",
    "The university provides career counseling and internship guidance for students."
]

query = "Is Python a Programming Language?"
print("User Question:", query)

### Step 4: Compute Document Embeddings

In [ ]:
document_embeddings = model.encode(documents)
print("Document Embeddings Shape:", document_embeddings.shape)

### Step 5: Load Generative Language Model (FLAN-T5-Small)

In [ ]:
print("Loading language model...")
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
generator = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Generative model loaded successfully!")

### Step 6: Define Document Retrieval and Prompt Building Functions

In [ ]:
def retrieve_documents(query, top_k=2):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, document_embeddings)[0]
    ranked_indices = np.argsort(similarities)[::-1]

    retrieved_documents = []
    for index in ranked_indices[:top_k]:
        retrieved_documents.append({
            "text": documents[index],
            "score": float(similarities[index])
        })
    return retrieved_documents

def create_prompt(query, retrieved_results):
    context = "\n".join(item["text"] for item in retrieved_results)

    prompt = f"""Answer the following question using the context below.
If the context doesn't have the answer, say "I don't have enough info to answer."

Context:
{context}

Question:
{query}

Answer:
"""
    return prompt

### Step 7: Retrieve Documents for Our Query

In [ ]:
retrieved_docs = retrieve_documents(query, top_k=2)
print("Retrieved Documents:")
for i, item in enumerate(retrieved_docs):
    print(f"{i+1}. {item['text']} (Similarity Score: {item['score']:.4f})")

### Step 8: Build Prompt and Generate Answer

In [ ]:
prompt = create_prompt(query, retrieved_docs)
print("Generated Prompt:\n", prompt)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = generator.generate(**inputs, max_new_tokens=150)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Model Output:", result)

### Step 9: Define the RAG Chatbot Loop function

In [ ]:
def rag_chatbot(query_input):
    docs = retrieve_documents(query_input, top_k=2)
    prompt_str = create_prompt(query_input, docs)
    inputs_t = tokenizer(prompt_str, return_tensors="pt")
    outputs_t = generator.generate(**inputs_t, max_new_tokens=150)
    ans = tokenizer.decode(outputs_t[0], skip_special_tokens=True)
    print("\n--- RAG Response ---")
    print(ans)
    print("---------------------\n")

### Step 10: Interactive Chat Loop (Optional Execution)

In [ ]:
while True:
    user_question = input("Enter your question (or type 'exit' to quit): ")
    if not user_question.strip():
        print("Please enter a valid question")
        continue
    if user_question.strip().lower() == 'exit':
        print("Goodbye!")
        break

    rag_chatbot(user_question)

### Step 11: Load PDFs as the Knowledge Base

In [ ]:
%pip install pypdf

import os
from pypdf import PdfReader

# Colab keeps uploaded files in /content; locally, put the PDFs next to this notebook
pdf_folder = "/content" if os.path.isdir("/content") else "."
pdf_files = [
    os.path.join(pdf_folder, "Guidelines-College-SPOC.pdf"),
    os.path.join(pdf_folder, "NeuraForge_SIH26024_Idea_Submission.pdf")
]

def chunk_text(text, chunk_size=80, overlap=20):
    # all-MiniLM-L6-v2 reads at most 256 tokens and flan-t5 512 (two chunks + question),
    # so a whole page is too long; 80 words stays under both limits
    words = text.split()
    chunks = []
    for start in range(0, len(words), chunk_size - overlap):
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break
    return chunks

new_documents = []

for pdf_path in pdf_files:
    if os.path.exists(pdf_path):
        print(f"Extracting text from: {pdf_path}")
        reader = PdfReader(pdf_path)
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                for chunk in chunk_text(text):
                    if len(chunk) > 30:
                        new_documents.append(f"[{os.path.basename(pdf_path)} - Page {page_num+1}] {chunk}")
    else:
        print(f"File not found: {pdf_path}")

print(f"Successfully parsed {len(new_documents)} chunks from PDFs.")

if new_documents:
    documents = new_documents
    document_embeddings = model.encode(documents)
    print("Document Embeddings shape updated to:", document_embeddings.shape)
else:
    print("No PDF text found, keeping the sample documents.")

### Step 12: Chat with the PDFs

In [ ]:
while True:
    user_question = input("Enter your question (or type 'exit' to quit): ")
    if not user_question.strip():
        print("Please enter a valid question")
        continue
    if user_question.strip().lower() == 'exit':
        print("Goodbye!")
        break

    rag_chatbot(user_question)